# Silver Layer Demo

This notebook demonstrates that the local Bronze to Silver pipeline has already processed the collected weather data.

It does not run the pipeline again. It only reads the existing local `data/processed/silver` output and shows evidence that the normalized Silver Layer exists.

In [1]:
from pathlib import Path
import json
from collections import Counter

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Silver Layer":
    PROJECT_ROOT = PROJECT_ROOT.parent

SILVER_ROOT = PROJECT_ROOT / "data" / "processed" / "silver" / "domain=weather"
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze" / "domain=weather"

print("Project root:", PROJECT_ROOT)
print("Bronze exists:", BRONZE_ROOT.exists(), BRONZE_ROOT)
print("Silver exists:", SILVER_ROOT.exists(), SILVER_ROOT)

Project root: E:\Academy of Mohyla\Course3_Stage2\bigData
Bronze exists: True E:\Academy of Mohyla\Course3_Stage2\bigData\data\bronze\domain=weather
Silver exists: True E:\Academy of Mohyla\Course3_Stage2\bigData\data\processed\silver\domain=weather


## 1. Discover Silver Files

The Silver Layer is stored locally under `data/processed/silver/domain=weather`.
The path contains `source`, `schema_v=1`, randomized `batch_id`, `ingest_date`, and `hour` partitions.

In [2]:
silver_files = sorted(SILVER_ROOT.rglob("*.jsonl"))
print("Silver JSONL files:", len(silver_files))

for path in silver_files[:10]:
    print(path.relative_to(PROJECT_ROOT))

Silver JSONL files: 10
data\processed\silver\domain=weather\source=open_meteo_api\schema_v=1\batch_id=204f374d-6537-4082-9e55-f9ec144c44de\ingest_date=2026-04-06\hour=13\part-204f374d-6537-4082-9e55-f9ec144c44de.jsonl
data\processed\silver\domain=weather\source=open_meteo_api\schema_v=1\batch_id=204f374d-6537-4082-9e55-f9ec144c44de\ingest_date=2026-04-06\hour=14\part-204f374d-6537-4082-9e55-f9ec144c44de.jsonl
data\processed\silver\domain=weather\source=open_meteo_api\schema_v=1\batch_id=204f374d-6537-4082-9e55-f9ec144c44de\ingest_date=2026-04-06\hour=15\part-204f374d-6537-4082-9e55-f9ec144c44de.jsonl
data\processed\silver\domain=weather\source=open_meteo_api\schema_v=1\batch_id=204f374d-6537-4082-9e55-f9ec144c44de\ingest_date=2026-04-21\hour=09\part-204f374d-6537-4082-9e55-f9ec144c44de.jsonl
data\processed\silver\domain=weather\source=open_meteo_api\schema_v=1\batch_id=ae3e678e-a69e-499e-92e6-e9a15d93b9fb\ingest_date=2026-04-06\hour=13\part-ae3e678e-a69e-499e-92e6-e9a15d93b9fb.jsonl
da

## 2. Load Silver Records

Here we load the normalized Silver records from JSONL files and display aggregate statistics.

In [3]:
records = []
for path in silver_files:
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                record = json.loads(line)
                record["silver_file"] = str(path.relative_to(PROJECT_ROOT))
                records.append(record)

print("Loaded silver rows:", len(records))
print("Sources:", dict(Counter(row["source"] for row in records)))
print("Authors:", sorted(set(row.get("author_surname") for row in records)))
print("Schema versions:", sorted(set(row.get("schema_v") for row in records)))
print("Batch IDs:", len(set(row.get("batch_id") for row in records)))

Loaded silver rows: 1640
Sources: {'open_meteo_api': 680, 'open_meteo_archive_api': 960}
Authors: ['Yermolovych']
Schema versions: [1]
Batch IDs: 3


## 3. Show Normalized Rows

The records below are already normalized: fields are extracted into columns instead of being stored as raw nested payload.

In [4]:
for row in records[:5]:
    print({
        "author_surname": row["author_surname"],
        "source": row["source"],
        "city": row["city"],
        "event_ts": row["event_ts"],
        "temperature_2m": row["temperature_2m"],
        "relative_humidity_2m": row["relative_humidity_2m"],
        "precipitation": row["precipitation"],
        "wind_speed_10m": row["wind_speed_10m"],
        "batch_id": row["batch_id"],
    })

{'author_surname': 'Yermolovych', 'source': 'open_meteo_api', 'city': 'Dnipro', 'event_ts': '2026-04-06 16:30:00', 'temperature_2m': 20.1, 'relative_humidity_2m': 29.0, 'precipitation': 0.0, 'wind_speed_10m': 19.5, 'batch_id': '204f374d-6537-4082-9e55-f9ec144c44de'}
{'author_surname': 'Yermolovych', 'source': 'open_meteo_api', 'city': 'Mariupol', 'event_ts': '2026-04-06 16:30:00', 'temperature_2m': 13.9, 'relative_humidity_2m': 42.0, 'precipitation': 0.0, 'wind_speed_10m': 14.7, 'batch_id': '204f374d-6537-4082-9e55-f9ec144c44de'}
{'author_surname': 'Yermolovych', 'source': 'open_meteo_api', 'city': 'Zaporizhzhia', 'event_ts': '2026-04-06 16:45:00', 'temperature_2m': 19.6, 'relative_humidity_2m': 28.0, 'precipitation': 0.0, 'wind_speed_10m': 18.2, 'batch_id': '204f374d-6537-4082-9e55-f9ec144c44de'}
{'author_surname': 'Yermolovych', 'source': 'open_meteo_api', 'city': 'Zaporizhzhia', 'event_ts': '2026-04-06 16:45:00', 'temperature_2m': 19.6, 'relative_humidity_2m': 28.0, 'precipitation':

## 4. Validate Silver Data

The Silver pipeline filters out nulls, invalid timestamps, invalid coordinates, invalid temperature values, invalid humidity, negative precipitation, and negative wind speed.

In [5]:
required_fields = [
    "author_surname", "domain", "source", "ingest_ts", "event_ts", "city",
    "latitude", "longitude", "temperature_2m", "relative_humidity_2m",
    "precipitation", "wind_speed_10m", "schema_v", "batch_id",
]

def is_valid(row):
    return (
        all(row.get(field) is not None for field in required_fields)
        and row["author_surname"] == "Yermolovych"
        and row["domain"] == "weather"
        and -90 <= row["latitude"] <= 90
        and -180 <= row["longitude"] <= 180
        and -100 <= row["temperature_2m"] <= 100
        and 0 <= row["relative_humidity_2m"] <= 100
        and row["precipitation"] >= 0
        and row["wind_speed_10m"] >= 0
    )

valid_count = sum(1 for row in records if is_valid(row))
invalid_count = len(records) - valid_count

print("Valid rows:", valid_count)
print("Invalid rows:", invalid_count)
print("Validation passed:", invalid_count == 0 and valid_count > 0)

Valid rows: 1640
Invalid rows: 0
Validation passed: True


## 5. Silver Output Summary

This section groups rows by source, city, and batch ID to show that data was processed into the Silver Layer.

In [6]:
print("Rows by source:")
for source, count in Counter(row["source"] for row in records).items():
    print(f"  {source}: {count}")

print("\nRows by city, first 10:")
for city, count in Counter(row["city"] for row in records).most_common(10):
    print(f"  {city}: {count}")

print("\nBatch IDs:")
for batch_id in sorted(set(row["batch_id"] for row in records)):
    print(" ", batch_id)

Rows by source:
  open_meteo_api: 680
  open_meteo_archive_api: 960

Rows by city, first 10:
  Dnipro: 164
  Mariupol: 164
  Zaporizhzhia: 164
  Odesa: 164
  Kryvyi Rih: 164
  Kyiv: 164
  Lviv: 164
  Kharkiv: 164
  Mykolaiv: 164
  Donetsk: 164

Batch IDs:
  204f374d-6537-4082-9e55-f9ec144c44de
  ae3e678e-a69e-499e-92e6-e9a15d93b9fb
  c02db826-d552-48ac-9dfc-8bfd4eea101a
